# Agentic Design Patterns: The Reflection Pattern

Welcome to Module 05! The first pattern we'll explore is **Reflection**. 

In simple workflows, an LLM generates a response and sends it directly to the user. This is akin to a writer publishing their very first draft without reviewing it. In complex tasks, like writing code or summarizing large texts, the first draft is rarely perfect.

The **Reflection Pattern** adds a critical feedback loop. The LLM acts as an "Actor" to generate the content, and then a "Reflector" (or Critic) to review its own work and suggest improvements. If the Reflector finds issues, it sends the feedback back to the Actor for a rewrite. This repeats until the Reflector approves or a maximum loop count is reached.

---

## 1. Environment Setup
We'll use LangGraph and Google Gemini to build our reflection loop.

In [ ]:
import os
from typing import Annotated, TypedDict, List
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from IPython.display import Image, display

load_dotenv()
# We use a slightly higher temperature for the Actor for creativity,
# and 0 temperature for the Reflector to ensure strict grading.
actor_llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.7)
reflector_llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

print("Reflection Environment Ready!")

## 2. Defining State & Pydantic Schema
We need to keep track of the conversation, the current generation, and how many times we've tried to improve it to prevent infinite loops.

In [ ]:
class ReflectionState(TypedDict):
    messages: Annotated[List[BaseMessage], lambda a, b: a + b]
    revision_number: int
    is_approved: bool

# Pydantic schema for the Reflector to use
class ReflectionCritique(BaseModel):
    """Critique the previous response and decide if it is approved."""
    feedback: str = Field(description="Detailed, actionable feedback on what is wrong or missing.")
    is_approved: bool = Field(description="True if the response fully satisfies the user request and is high quality, False otherwise.")

reflector_llm_structured = reflector_llm.with_structured_output(ReflectionCritique)

## 3. Defining the Nodes
The Actor writes. The Reflector critiques.

In [ ]:
def actor_node(state: ReflectionState):
    # Provide robust instructions
    system_msg = SystemMessage(content="""You are an expert Python developer. Generate the code requested by the user.
    If the user or a reviewer provides feedback, use it to improve and rewrite your code. DO NOT apologize or make excuses, just provide the improved code.
    Enclose your code in markdown blocks.""")
    
    # Increment revision count. If it doesn't exist, it's 0.
    current_rev = state.get("revision_number", 0) + 1
    
    response = actor_llm.invoke([system_msg] + state["messages"])
    return {
        "messages": [AIMessage(content=response.content, name="Actor")],
        "revision_number": current_rev
    }

def reflector_node(state: ReflectionState):
    # The Reflector checks the latest message
    system_msg = SystemMessage(content="""You are a strict, senior Lead Software Engineer.
    Review the Python code provided by the Actor.
    Check for:
    1. Correctness (does it solve the problem?)
    2. Edge cases (are inputs validated?)
    3. Code quality (docstrings, typing, readability)
    
    If the code fails any of these, set is_approved to False and provide EXTREMELY specific feedback.
    If it is perfect, set is_approved to True.""")
    
    critique = reflector_llm_structured.invoke([system_msg] + state["messages"])
    
    return {
        # We append the critique as a HumanMessage so the Actor sees it as a new prompt to fix the code
        "messages": [HumanMessage(content=f"Reviewer Feedback: {critique.feedback}", name="Reflector")],
        "is_approved": critique.is_approved
    }

# Routing Logic
def should_continue(state: ReflectionState):
    if state.get("is_approved", False):
        print("\n*** Reflector APPROVED the response! Ending loop. ***\n")
        return END
    
    # hard stop to prevent infinite loops
    if state.get("revision_number", 0) > 3:
        print("\n*** Max revisions reached. Forcing END. ***\n")
        return END
        
    print(f"\n*** Reflector REJECTED the response. Sending back to Actor (Revision {state.get('revision_number', 0) + 1})... ***\n")
    return "actor"

## 4. Compile and Visualize the Graph

In [ ]:
builder = StateGraph(ReflectionState)

builder.add_node("actor", actor_node)
builder.add_node("reflector", reflector_node)

builder.add_edge(START, "actor")
builder.add_edge("actor", "reflector")
builder.add_conditional_edges("reflector", should_continue)

app = builder.compile()

try:
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception:
    print("Graph visualization failed.")

## 5. Testing the Reflection Loop
Let's assign the Actor a task where a first draft is usually poor without explicitly requesting edge-case handling. The Reflector will likely catch it.

In [ ]:
query = "Write a Python function to divide two numbers."

initial_state = {
    "messages": [HumanMessage(content=query)],
    "revision_number": 0
}

# stream_mode="updates" yields the dictionary returned by each node
for chunk in app.stream(initial_state, stream_mode="updates"):
    for node_name, state_update in chunk.items():
        message = state_update.get("messages", [])[-1]
        print(f"\n--- View from Node: {node_name.upper()} ---")
        print(message.content)

## Summary

The Reflection pattern dramatically improves reliability.

Instead of hoping the LLM gets the prompt right the first time, you formalize a "Review phase." The model critiques itself against specific criteria. 
Because LLMs are often much better at *evaluating* code/text than *writing* it zero-shot, this leads to significantly higher quality outputs in production.